# Aligning five Visium sections to a common reference

Same idea as `examples/align_two_visium_sections.ipynb`, scaled up: instead of naming each
section's own variables by hand, sections live in a dict keyed by name, and one section is
picked as the common `reference` that every other section gets registered against
independently (same reasoning as that notebook's ST50: each pairwise registration is its own,
since elastic transforms can't be chained/inverted). Adding a sixth section later just means
adding one more entry to `section_paths` below -- nothing else about the notebook changes.

The five sections are separate consecutive mouse brain sections from the same public dataset
as the two-section notebook.

In [1]:
from visium_utils import load_classic_visium_hires  # examples/visium_utils.py

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns

import spatialwarp as sw

/home/croizer/miniconda3/envs/msi/lib/python3.13/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/croizer/miniconda3/envs/msi/lib/python3.13/site-packages/spatialdata/_core/query/relational_query.py:504: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  left = partial(_left_join_spatialelement_table)
/home/croizer/miniconda3/envs/msi/lib/python3.13/site-packages/spatialdata/_core/query/relational_query.py:505: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  left_exclusive = partial(_left_exclusive_join_

A couple of steps below open a window where you click points with your mouse. For that to
work in Jupyter, run this cell first to switch to a real window backend instead of the default
static-image mode (`tk` works out of the box with no extra install; use `qt` instead if you
have PyQt5/PySide installed and prefer it).

In [2]:
%matplotlib tk

## Step 1 -- Load all five sections

`section_paths` maps a short name to its Space Ranger `outs/` folder and counts-file prefix (all
five live under the same base directory, with dataset-id-prefixed counts files rather than Space
Ranger's default `filtered_feature_bc_matrix.h5`, hence `counts_file=`). `load_classic_visium_hires`
(see `examples/visium_utils.py`) rescales each section's own spot coordinates into its own hires
image's pixel space, exactly as in the two-section notebook.

In [3]:
base_path = "/home/croizer/Documents/01_Ressources/02_Public_Visium/brain/mice/cell2loc"
section_paths = {
    "ST48": "ST8059048",
    "ST49": "ST8059049",
    "ST50": "ST8059050",
    "ST51": "ST8059051",
    "ST52": "ST8059052",
}

sections = {}
for name, prefix in section_paths.items():
    sdata, he = load_classic_visium_hires(
        f"{base_path}/{name}", dataset_id=name, counts_file=f"{prefix}_filtered_feature_bc_matrix.h5"
    )
    sections[name] = (sdata, he)
    print(f"{name}: {sdata.tables['visium'].n_obs} spots, image {he.shape}")

INFO     reading                                                                                                   
         /home/croizer/Documents/01_Ressources/02_Public_Visium/brain/mice/cell2loc/ST48/ST8059048_filtered_feature
         _bc_matrix.h5                                                                                             


/home/croizer/miniconda3/envs/msi/lib/python3.13/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/croizer/miniconda3/envs/msi/lib/python3.13/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/croizer/miniconda3/envs/msi/lib/python3.13/site-packages/spatialdata/models/models.py:1035: UserWarning: Converting `region_key: region` to categorical dtype.
  return convert_region_column_to_categorical(adata)


INFO     no axes information specified in the object, setting `dims` to: ('c', 'y', 'x')                           
ST48: 2987 spots, image (2000, 1969, 3)
INFO     reading                                                                                                   
         /home/croizer/Documents/01_Ressources/02_Public_Visium/brain/mice/cell2loc/ST49/ST8059049_filtered_feature
         _bc_matrix.h5                                                                                             


/home/croizer/miniconda3/envs/msi/lib/python3.13/site-packages/spatialdata/_core/spatialdata.py:158: UserWarning: The table is annotating 'ST48', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
/home/croizer/miniconda3/envs/msi/lib/python3.13/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/croizer/miniconda3/envs/msi/lib/python3.13/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/croizer/miniconda3/envs/msi/lib/python3.13/site-packages/spatialdata/models/models.py:1035: UserWarning: Converting `region_key: region` to categorical dtype.
  return convert_region_column_to_categorical(adata)


INFO     no axes information specified in the object, setting `dims` to: ('c', 'y', 'x')                           
ST49: 3499 spots, image (2000, 1969, 3)
INFO     reading                                                                                                   
         /home/croizer/Documents/01_Ressources/02_Public_Visium/brain/mice/cell2loc/ST50/ST8059050_filtered_feature
         _bc_matrix.h5                                                                                             


...

/home/croizer/miniconda3/envs/msi/lib/python3.13/site-packages/spatialdata/_core/spatialdata.py:158: UserWarning: The table is annotating 'ST50', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
/home/croizer/miniconda3/envs/msi/lib/python3.13/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/croizer/miniconda3/envs/msi/lib/python3.13/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/croizer/miniconda3/envs/msi/lib/python3.13/site-packages/spatialdata/models/models.py:1035: UserWarning: Converting `region_key: region` to categorical dtype.
  return convert_region_column_to_categorical(adata)


INFO     no axes information specified in the object, setting `dims` to: ('c', 'y', 'x')                           
ST51: 2409 spots, image (2000, 1963, 3)
INFO     reading                                                                                                   
         /home/croizer/Documents/01_Ressources/02_Public_Visium/brain/mice/cell2loc/ST52/ST8059052_filtered_feature
         _bc_matrix.h5                                                                                             


/home/croizer/miniconda3/envs/msi/lib/python3.13/site-packages/spatialdata/_core/spatialdata.py:158: UserWarning: The table is annotating 'ST51', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
/home/croizer/miniconda3/envs/msi/lib/python3.13/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/croizer/miniconda3/envs/msi/lib/python3.13/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/croizer/miniconda3/envs/msi/lib/python3.13/site-packages/spatialdata/models/models.py:1035: UserWarning: Converting `region_key: region` to categorical dtype.
  return convert_region_column_to_categorical(adata)


INFO     no axes information specified in the object, setting `dims` to: ('c', 'y', 'x')                           
ST52: 2576 spots, image (2000, 1950, 3)


/home/croizer/miniconda3/envs/msi/lib/python3.13/site-packages/spatialdata/_core/spatialdata.py:158: UserWarning: The table is annotating 'ST52', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)


## Step 2 -- Register every other section against the reference

`reference` is the one section every other section gets registered and matched against; any of
the five would work equally well as the anchor. For each other section: click landmarks (a
window opens per section, in turn -- close it to move to the next one), register, and match --
the exact same `sw.pick_landmarks` / `sw.register_elastic` / `sw.align` calls as the two-section
notebook, just looped.

Each section's matched expression is merged into one running `merged` object, keyed as
`obsm["<name>_expression"]`, reindexed onto the first section's matched spots (since each
pairwise match can keep a slightly different subset of the reference's spots) -- same pattern
as the two-section notebook's ST50 step, generalized to however many other sections there are.

In [4]:
reference = "ST49"
ref_sdata, ref_he = sections[reference]

merged = None
registration_results = {}

for name, (sdata, he) in sections.items():
    if name == reference:
        continue

    moving_landmarks, fixed_landmarks, _, _ = sw.pick_landmarks(
        he, ref_he, output_csv=f"landmarks_{name}_{reference}.csv"
    )

    registration_result = sw.register_elastic(
        moving_image=he,
        fixed_image=ref_he,
        moving_landmarks=moving_landmarks,
        fixed_landmarks=fixed_landmarks,
        mesh_size=(8, 8),
        number_of_iterations=100,
    )
    registration_result.save(f"registration_{name}_{reference}")
    registration_results[name] = registration_result

    result = sw.align(
        moving=sdata,
        fixed=ref_sdata,
        registration_result=registration_result,
        distance_threshold=15.0,
        moving_table_key="visium",
        fixed_table_key="visium",
        moving_obsm_key=f"{name}_expression",
    )
    print(f"{name} vs {reference}: kept {result.n_obs} of {ref_sdata.tables['visium'].n_obs} {reference} spots")

    if merged is None:
        merged = result
    else:
        merged.obsm[f"{name}_expression"] = result.obsm[f"{name}_expression"].reindex(merged.obs_names)

print(f"\nfinal merged: {merged.n_obs} {reference} spots, sections: {list(registration_results)}")
merged

ST48 vs ST49: kept 3330 of 3499 ST49 spots
ST50 vs ST49: kept 3162 of 3499 ST49 spots
ST51 vs ST49: kept 3427 of 3499 ST49 spots
ST52 vs ST49: kept 3421 of 3499 ST49 spots

final merged: 3330 ST49 spots, sections: ['ST48', 'ST50', 'ST51', 'ST52']


AnnData object with n_obs × n_vars = 3330 × 31053
    obs: 'in_tissue', 'array_row', 'array_col', 'spot_id', 'region', 'nearest_index', 'nearest_distance'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial', 'spatialdata_attrs', 'spatialwarp'
    obsm: 'spatial', 'spatial_warped', 'ST48_expression', 'ST50_expression', 'ST51_expression', 'ST52_expression'

## Step 3 -- QC: total-count agreement across all sections

Same sanity check as the two-section notebook (two sections of the same tissue should broadly
agree on where RNA density is high/low, even without spot-for-spot alignment), generalized to a
correlation matrix over all five sections instead of a single pair.

In [5]:
def _dense_1d(x):
    return x.toarray().ravel() if sp.issparse(x) else np.asarray(x).ravel()

total_counts = {reference: _dense_1d(merged.X.sum(axis=1))}
for name in registration_results:
    total_counts[name] = merged.obsm[f"{name}_expression"].sum(axis=1).values

total_counts_df = pd.DataFrame(total_counts, index=merged.obs_names)
corr_matrix = total_counts_df.corr()

sns.clustermap(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1, annot=True, fmt=".2f", figsize=(6, 6))
plt.show()

## Step 4 -- Same gene, all five sections, side by side

With more than two sections, an RGB additive overlay runs out of channels -- a small-multiples
grid of spatial plots scales to any number of sections instead. All five panels share the same
color scale for a fair comparison, plotted at `merged`'s own (reference) spot positions.

In [17]:
gene = "Mbp"
spatial_coords = merged.obsm["spatial"]

values_per_section = {reference: _dense_1d(merged[:, gene].X)}
for name in registration_results:
    values_per_section[name] = merged.obsm[f"{name}_expression"][gene].values

all_values = np.concatenate(list(values_per_section.values()))
vmax = np.nanpercentile(all_values, 99)

fig, axes = plt.subplots(1, len(values_per_section), figsize=(4 * len(values_per_section), 4.5))
for ax, (name, values) in zip(axes, values_per_section.items()):
    sca = ax.scatter(spatial_coords[:, 0], -spatial_coords[:, 1], c=values, cmap="viridis", vmin=0, vmax=vmax, s=6)
    ax.set_title(name)
    ax.set_aspect("equal")
    ax.axis("off")
fig.colorbar(sca, ax=axes, shrink=0.6, label=gene)
plt.show()

## Step 5 -- 3D "stacked sheets", all five sections

Every section's matched expression is already reindexed onto `merged`'s own spots (Steps 2-3),
so all five layers share the exact same `(array_col, array_row)` grid position by construction --
nothing left to warp, only `z` (which section) and color (that section's expression) change
between sheets. Same reasoning as the two-section notebook's 3D bonus plot, generalized to five
sheets via a loop instead of one `ax.scatter` call per section.

In [7]:
# On some machines, an old system-wide matplotlib install's mpl_toolkits shim registers a
# broken, version-mismatched mpl_toolkits in sys.modules before this notebook even starts
# (site.py runs it at interpreter startup), which also makes matplotlib.projections give up on
# registering the "3d" projection at its own (one-time) import. Evicting the stale cache entry
# and re-registering Axes3D by hand fixes both -- harmless if your environment doesn't have
# this issue.
import sys

for _name in list(sys.modules):
    if _name == "mpl_toolkits" or _name.startswith("mpl_toolkits."):
        del sys.modules[_name]

from mpl_toolkits.mplot3d import Axes3D

import matplotlib.projections as mprojections

if "3d" not in mprojections.get_projection_names():
    mprojections.register_projection(Axes3D)

In [16]:
gene = "Mbp"
grid_xy = merged.obs[["array_col", "array_row"]].values.astype(float)

section_names = [reference] + list(registration_results)
z_levels = {name: i * 500 for i, name in enumerate(section_names)}

values_per_section = {reference: _dense_1d(merged[:, gene].X)}
for name in registration_results:
    values_per_section[name] = merged.obsm[f"{name}_expression"][gene].fillna(0).values

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="3d")
for name in section_names:
    ax.scatter(
        grid_xy[:, 0], -grid_xy[:, 1], zs=z_levels[name],
        c=values_per_section[name], cmap="viridis", s=4,
    )
ax.set_zticks(list(z_levels.values()))
ax.set_zticklabels(list(z_levels.keys()))
ax.set_title(f"{gene} across all five sections")
plt.show()

In [37]:
gene = "Opalin"
grid_xy = merged.obs[["array_col", "array_row"]].values.astype(float)

section_names = [reference] + list(registration_results)
z_levels = {name: i * 500 for i, name in enumerate(section_names)}

values_per_section = {reference: _dense_1d(merged[:, gene].X)}
for name in registration_results:
    values_per_section[name] = merged.obsm[f"{name}_expression"][gene].fillna(0).values

all_values = np.concatenate(list(values_per_section.values()))
vmax = np.nanpercentile(all_values, 99)
percentile_floor = 1
threshold = np.nanpercentile(all_values, percentile_floor)

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="3d")
for name in section_names:
    values = values_per_section[name]
    keep = values >= threshold
    ax.scatter(
        grid_xy[keep, 0], -grid_xy[keep, 1], zs=z_levels[name],
        c=values[keep], cmap="coolwarm", vmin=0, vmax=vmax, s=15,
        depthshade=False,
    )
ax.set_zticks(list(z_levels.values()))
ax.set_zticklabels(list(z_levels.keys()))
ax.view_init(elev=20, azim=-60)
ax.set_box_aspect(None, zoom=1.3)
plt.show()
